# 🎨 ROTBOTS — Effects & Chain Log

---

Assign FFmpeg effects per AI scene. View the chain of AI decisions.

Skip if `ENABLE_EFFECTS = False` in your plan.

---

In [ ]:
# SETUP
import json, random
from pathlib import Path
from IPython.display import display, Markdown, HTML
from google.colab import drive
drive.mount('/content/drive')
BASE_DIR = Path('/content/drive/MyDrive/rotbots_workshop')
print('✅ Setup')

In [ ]:
# LOAD SESSION
sessions = sorted([d.name for d in BASE_DIR.iterdir() if d.is_dir() and (d/'video_plan.json').exists()])
if not sessions: print('❌ No plan! Run 01_Video_Plan first.')
else:
    for i,s in enumerate(sessions): print(f'   {i}: {s}')
SESSION_NAME = sessions[-1] if sessions else ''
SESSION_DIR = BASE_DIR / SESSION_NAME
with open(SESSION_DIR/'video_plan.json') as f: plan = json.load(f)
print(f'✅ Session: {SESSION_NAME}')

In [ ]:
# EFFECTS
EFFECTS = {'none':'No effect','film_grain':'Subtle grain','vhs_artifacts':'VHS tracking','celluloid_scratches':'Film scratches','sepia_tone':'Warm sepia','bw_transition':'Black & white','color_grade_warm':'Warm orange','color_grade_cool':'Cool blue','vignette':'Dark edges','flicker':'Projector flicker','desaturate':'Muted colors'}
for k,v in EFFECTS.items(): print(f'   {k:25s} {v}')

In [ ]:
# ASSIGN EFFECTS
EFFECT_MODE = 'random'
PER_SCENE_EFFECTS = {}
EFFECT_INTENSITY = 0.7
pf = SESSION_DIR / 'prompts.json'
if not pf.exists(): print('❌ No prompts.json!')
elif not plan.get('enable_effects',True): print('🎨 Effects disabled.')
else:
    with open(pf) as f: prompts = json.load(f)
    enames = [k for k in EFFECTS if k!='none']
    for p in prompts:
        sn=p['scene']
        if sn in PER_SCENE_EFFECTS: p['ffmpeg_effects']=[PER_SCENE_EFFECTS[sn]] if PER_SCENE_EFFECTS[sn]!='none' else []
        elif EFFECT_MODE=='random': p['ffmpeg_effects']=[random.choice(enames)]
        elif EFFECT_MODE!='none': p['ffmpeg_effects']=[EFFECT_MODE]
        else: p['ffmpeg_effects']=[]
        p['effect_intensity']=EFFECT_INTENSITY
    with open(pf,'w') as f: json.dump(prompts,f,indent=2)
    print('✅ Effects:')
    for p in prompts: print(f'   Scene {p["scene"]}: {p.get("ffmpeg_effects",[])} ({EFFECT_INTENSITY:.0%})')

---
## 📊 Chain Log

In [ ]:
# CHAIN LOG
chain={'session':SESSION_NAME,'config':plan,'interactions':[]}
if (SESSION_DIR/'summaries.json').exists():
    with open(SESSION_DIR/'summaries.json') as f: d=json.load(f)
    for s in d.get('sources',[]): chain['interactions'].append({'stage':'summarize','purpose':s['source'][:40],'length':len(s['summary'])})
if (SESSION_DIR/'essay_script.json').exists():
    with open(SESSION_DIR/'essay_script.json') as f: e=json.load(f)
    chain['interactions'].append({'stage':'outline','purpose':e.get('title','')})
    for ch in e.get('chapters',[]):
        for sec in ch.get('sections',[]): chain['interactions'].append({'stage':'chapter','purpose':f'Ch{ch["chapter"]}.{sec.get("section","?")}'})
if (SESSION_DIR/'prompts.json').exists():
    with open(SESSION_DIR/'prompts.json') as f: pd=json.load(f)
    for p in pd: chain['interactions'].append({'stage':'t2v','purpose':f'Scene {p["scene"]}','effect':p.get('ffmpeg_effects',[])})
with open(SESSION_DIR/'chain_log.json','w') as f: json.dump(chain,f,indent=2)
print(f'📊 {len(chain["interactions"])} decisions')
for i in chain['interactions']:
    eff=f' 🎨{i["effect"][0]}' if i.get('effect') else ''
    print(f'   [{i["stage"]}] {i["purpose"]}{eff}')

---
*ROTBOTS Workshop — LI-MA 2026*